In [2]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()

# =========================================================
# TABLE NAMES
# =========================================================
OUT_DAILY_NAIVE = "a2_daily_sales_by_region_naive"
OUT_TOP10_NAIVE = "a2_top_10_products_by_category_naive"
OUT_CLV_NAIVE = "a2_customer_lifetime_value_naive"
OUT_RETURN_RATE_NAIVE = "a2_monthly_return_rate_naive"

OUT_DAILY_OPT = "a2_daily_sales_by_region_optimized"
OUT_TOP10_OPT = "a2_top_10_products_by_category_optimized"
OUT_CLV_OPT = "a2_customer_lifetime_value_optimized"
OUT_RETURN_RATE_OPT = "a2_monthly_return_rate_optimized"

StatementMeta(, 3c222c5c-b862-42a0-b8da-6e99f3878232, 4, Finished, Available, Finished, False)

In [3]:
def safe_read(table_name: str):
    """Read a table and raise a clear error if missing."""
    try:
        return spark.read.table(table_name)
    except Exception as exc:
        raise RuntimeError(f"Unable to read table '{table_name}'. Verify the notebook completed successfully.") from exc


def compare_counts(label: str, naive_table: str, opt_table: str) -> None:
    naive_df = safe_read(naive_table)
    opt_df = safe_read(opt_table)

    naive_count = naive_df.count()
    opt_count = opt_df.count()
    status = "PASS" if naive_count == opt_count else "FAIL"

    print(f"[{status}] {label} row count -> naive={naive_count}, optimized={opt_count}")


def compare_sum(label: str, naive_table: str, opt_table: str, metric_col: str) -> None:
    naive_df = safe_read(naive_table)
    opt_df = safe_read(opt_table)

    naive_total = naive_df.agg(F.round(F.sum(metric_col), 2).alias("total")).collect()[0]["total"]
    opt_total = opt_df.agg(F.round(F.sum(metric_col), 2).alias("total")).collect()[0]["total"]
    status = "PASS" if naive_total == opt_total else "FAIL"

    print(f"[{status}] {label} sum({metric_col}) -> naive={naive_total}, optimized={opt_total}")


def compare_top10_distribution() -> None:
    naive_df = safe_read(OUT_TOP10_NAIVE)
    opt_df = safe_read(OUT_TOP10_OPT)

    naive_counts = (
        naive_df.groupBy("category")
        .count()
        .orderBy("category")
    )

    opt_counts = (
        opt_df.groupBy("category")
        .count()
        .orderBy("category")
    )

    diff_count = (
        naive_counts.alias("n")
        .join(opt_counts.alias("o"), on="category", how="full")
        .select(
            F.col("category"),
            F.coalesce(F.col("n.count"), F.lit(0)).alias("naive_count"),
            F.coalesce(F.col("o.count"), F.lit(0)).alias("optimized_count")
        )
        .filter(F.col("naive_count") != F.col("optimized_count"))
        .count()
    )

    status = "PASS" if diff_count == 0 else "FAIL"
    print(f"[{status}] Top 10 product distribution by category validation")


def main() -> None:
    print("=" * 70)
    print("ASSESSMENT 2 VALIDATION OUTPUT")
    print("=" * 70)

    print("\n1. ROW COUNT VALIDATION")
    compare_counts("Daily Sales by Region", OUT_DAILY_NAIVE, OUT_DAILY_OPT)
    compare_counts("Top 10 Products by Category", OUT_TOP10_NAIVE, OUT_TOP10_OPT)
    compare_counts("Customer Lifetime Value", OUT_CLV_NAIVE, OUT_CLV_OPT)
    compare_counts("Monthly Return Rate", OUT_RETURN_RATE_NAIVE, OUT_RETURN_RATE_OPT)

    print("\n2. AGGREGATE VALIDATION")
    compare_sum("Daily Sales by Region", OUT_DAILY_NAIVE, OUT_DAILY_OPT, "daily_sales_amount")
    compare_sum("Customer Lifetime Value", OUT_CLV_NAIVE, OUT_CLV_OPT, "customer_lifetime_value")

    print("\n3. DISTRIBUTION VALIDATION")
    compare_top10_distribution()

    print("\nValidation completed.")


if __name__ == "__main__":
    main()


StatementMeta(, 3c222c5c-b862-42a0-b8da-6e99f3878232, 5, Finished, Available, Finished, False)

ASSESSMENT 2 VALIDATION OUTPUT

1. ROW COUNT VALIDATION
[PASS] Daily Sales by Region row count -> naive=6570, optimized=6570
[PASS] Top 10 Products by Category row count -> naive=50, optimized=50
[PASS] Customer Lifetime Value row count -> naive=1000000, optimized=1000000
[PASS] Monthly Return Rate row count -> naive=36, optimized=36

2. AGGREGATE VALIDATION
[PASS] Daily Sales by Region sum(daily_sales_amount) -> naive=15303128809.63, optimized=15303128809.63
[PASS] Customer Lifetime Value sum(customer_lifetime_value) -> naive=15303128809.63, optimized=15303128809.63

3. DISTRIBUTION VALIDATION
[PASS] Top 10 product distribution by category validation

Validation completed.
